# California Housing Price Prediction

This notebook demonstrates California house price prediction using linear regression (which we'll go into more later in the semester). The goal of this notebook is to provide practice with:
* Data loading and understanding
* Data cleaning for tabular data
* Feature engineering for tabular data
* The train-test-validation split
* Data pre-processing and standardization
* Regression model training and evaluation

## Think about the ML Problem

You are tasked with pricing homes at a fair market value for homes in California. You are given the California Housing dataset to train your model. Before looking ahead at the rest of this notebook, how would you approach this problem?

YOUR ANSWER HERE

## 1. Setup and Data Loading

In [1]:
# Import all necessary libraries
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
import requests
import os

warnings.filterwarnings('ignore')

# Create data directory if it doesn't exist
os.makedirs('data', exist_ok=True)

# Download the California Housing dataset if it doesn't exist
dataset_path = 'data/housing.csv'
if not os.path.exists(dataset_path):
    print("Downloading California Housing dataset...")
    url = "https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv"
    
    # Download and save the file
    response = requests.get(url)
    if response.status_code == 200:
        with open(dataset_path, 'wb') as f:
            f.write(response.content)
        print("Dataset downloaded successfully!")
    else:
        print(f"Failed to download dataset. Status code: {response.status_code}")
        exit(1)

# Set default color scheme for consistency
default_colors = {'main': '#1f77b4', 'secondary': '#ff7f0e'}

Dataset downloaded successfully!


In [2]:
# Load the California Housing dataset
df = pd.read_csv('data/housing.csv')

print('Loaded the California Housing dataset!')
print(df.head())

Loaded the California Housing dataset!
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  


## 2. Data Understanding

Let's follow the lecture's "How to Look at the Data" framework:

### 2.1. How many data points and features are in our dataset?

In [3]:
# YOUR CODE HERE
num_samples = df.shape[0]
num_features = df.shape[1]

print(f"We have N={num_samples} samples in our dataset.")
print(f"Each sample has D={num_features} features (excluding target variable).")

We have N=20640 samples in our dataset.
Each sample has D=10 features (excluding target variable).


### 2.2. What are our features and target variable?

In [4]:
# Create visualization

# Set default color scheme for consistency
default_colors = {'main': '#1f77b4', 'secondary': '#ff7f0e'}

# Analyze feature types
print("Feature Types:")
print("-" * 40)
for col in df.columns:
    dtype = df[col].dtype
    unique_count = df[col].nunique()
    if pd.api.types.is_numeric_dtype(dtype):
        print(f"{col:20} Numeric   ({unique_count} unique values)")
    else:
        print(f"{col:20} Categorical ({unique_count} unique values)")

Feature Types:
----------------------------------------
longitude            Numeric   (844 unique values)
latitude             Numeric   (862 unique values)
housing_median_age   Numeric   (52 unique values)
total_rooms          Numeric   (5926 unique values)
total_bedrooms       Numeric   (1923 unique values)
population           Numeric   (3888 unique values)
households           Numeric   (1815 unique values)
median_income        Numeric   (12928 unique values)
median_house_value   Numeric   (3842 unique values)
ocean_proximity      Categorical (5 unique values)


### 2.3. What is the distribution of our target variable?

In [9]:
# Import visualization libraries
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Create interactive histogram with mean and median lines
mean_value = df['median_house_value'].mean()
median_value = df['median_house_value'].median()
fig = go.Figure()

# Add histogram
fig.add_trace(go.Histogram(
    x=df['median_house_value'],
    nbinsx=50,
    name='Distribution',
    marker_color=default_colors['main']
))

# Add mean line
fig.add_vline(x=mean_value, line_dash="dash", line_color="red",
              annotation_text=f"Mean: ${mean_value:,.0f}",
              annotation_yshift=10)

# Add median line
fig.add_vline(x=median_value, line_dash="dash", line_color="green",
              annotation_text=f"Median: ${median_value:,.0f}",
              annotation_yshift=-10)

# Update layout
fig.update_layout(
    title='Distribution of Target Variable (House Values)',
    title_x=0.5,
    xaxis_title='Median House Value ($)',
    yaxis_title='Count',
    template='plotly_white'
)

fig.show()

Using the interactive histogram above:
1. What is the shape of the distribution? Is it symmetric?
2. How do the mean and median values compare?
3. Are there any notable gaps or clusters in the distribution?
4. What might the distribution shape tell us about the housing market?

### 2.4. Are there any missing values or data quality issues to address?

Before handling outliers, let's check for and handle any missing values in our dataset.

In [11]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing values in our dataset:")
print("-" * 40)
for col, missing in missing_values.items():
    if missing > 0:
        print(f"{col}: {missing} ({missing/len(df)*100:.1f}%)")
    else:
        print(f"{col}: No missing values")

# Create visualization of missing values
missing_data = pd.DataFrame({
    'Feature': missing_values.index,
    'Missing Values': missing_values.values,
    'Percentage': (missing_values.values / len(df)) * 100
})

# Sort by number of missing values
missing_data = missing_data.sort_values('Missing Values', ascending=True)

# Create bar plot
fig = go.Figure()

fig.add_trace(go.Bar(
    y=missing_data['Feature'],
    x=missing_data['Percentage'],
    orientation='h',
    marker_color='lightcoral'
))

# Update layout
fig.update_layout(
    title='Missing Values by Feature',
    title_x=0.5,
    xaxis_title='Percentage of Missing Values',
    yaxis_title='Feature',
    template='plotly_white'
)

fig.show()

Missing values in our dataset:
----------------------------------------
longitude: No missing values
latitude: No missing values
housing_median_age: No missing values
total_rooms: No missing values
total_bedrooms: 207 (1.0%)
population: No missing values
households: No missing values
median_income: No missing values
median_house_value: No missing values
ocean_proximity: No missing values


Let's handle the missing values. In this exercise, we will choose to handle missing values through imputation.
1. For numerical features, we'll use median imputation.
2. For categorical features, we'll use mode imputation.

In [12]:
# Handle missing values
for column in df.columns:
    if df[column].isnull().any():
        if pd.api.types.is_numeric_dtype(df[column]):
            # Impute with median
            df.loc[df[column].isna(), column] = df[column].median()
            print(f"Imputed {column} with median value")
        else:
            # Impute with mode
            df.loc[df[column].isna(), column] = df[column].mode()[0]
            print(f"Imputed {column} with mode value")

# Verify no missing values remain
missing_after = df.isnull().sum()
if missing_after.sum() == 0:
    print("\nAll missing values have been handled successfully!")
else:
    print("\nWarning: Some missing values remain!")
    print(missing_after[missing_after > 0])

Imputed total_bedrooms with median value

All missing values have been handled successfully!


### 2.5. Are there any outliers or censored values we should handle?

In [13]:
# Analyze censored values
censored_count = len(df[df['median_house_value'] >= 500000])
total_count = len(df)
print(f"Censored Values Analysis:")
print(f"Number of houses valued at $500,000+: {censored_count}")
print(f"Percentage of dataset: {(censored_count/total_count)*100:.1f}%")

# Create before/after cleaning comparison
fig = make_subplots(rows=1, cols=2, subplot_titles=['Before Removing Censored Values', 
                                                   'After Removing Censored Values'])

# Before cleaning
fig.add_trace(
    go.Histogram(x=df['median_house_value'], nbinsx=50, name='Before',
                 marker_color=default_colors['main']),
    row=1, col=1
)

# Remove censored values
print("\nRemoving censored values...")
# YOUR CODE HERE
df = df[df['median_house_value'] < 500000]

# After cleaning
fig.add_trace(
    go.Histogram(x=df['median_house_value'], nbinsx=50, name='After',
                 marker_color=default_colors['secondary']),
    row=1, col=2
)

# Update layout
fig.update_layout(
    title_text='Effect of Removing Censored Values',
    title_x=0.5,
    showlegend=False,
    height=500,
    template='plotly_white'
)

# Update axes
for i in [1, 2]:
    fig.update_xaxes(title_text='Median House Value ($)', row=1, col=i)
    fig.update_yaxes(title_text='Count', row=1, col=i)

fig.show()

Censored Values Analysis:
Number of houses valued at $500,000+: 992
Percentage of dataset: 4.8%

Removing censored values...


Using the interactive histograms above:
1. What is the most common house value range before cleaning? (Use hover to see exact values)
2. How does the distribution shape change after removing censored values?
3. Are there any notable gaps or clusters in the distribution?
4. What percentage of data points were removed? Is this acceptable?

### 2.6. What features might we want to create to help predict house prices?

Before looking at the solution, think about what features might be useful for predicting house prices. Consider:
1. What ratios or combinations of existing features might be meaningful?
2. How can we better represent location information?
3. What domain knowledge about housing markets could inform feature creation?

YOUR ANSWER HERE

### 2.7. Implement the household-based features

Create features that capture the relationships between rooms, bedrooms, households, and population.
Be sure to handle any potential division by zero or infinity cases.

In [14]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [26]:
import numpy as np
# YOUR CODE HERE
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']
df['rooms_per_person'] = df['total_rooms'] / df['population']

for col in ['rooms_per_household', 'bedrooms_per_room', 'population_per_household', 'rooms_per_person']:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    df.loc[df[col].isna(), col] = df[col].median()

# Show statistics of new features
print("Statistics of engineered features:")
print("-" * 40)
print(df[['rooms_per_household', 'bedrooms_per_room', 
          'population_per_household', 'rooms_per_person']].describe())

# Create interactive box plots for the engineered features
fig = go.Figure()

for feature in ['rooms_per_household', 'bedrooms_per_room', 
                'population_per_household', 'rooms_per_person']:
    fig.add_trace(go.Box(
        y=df[feature],
        name=feature,
        boxpoints='outliers'
    ))

fig.update_layout(
    title='Distribution of Engineered Features',
    title_x=0.5,
    yaxis_title='Value',
    template='plotly_white',
    showlegend=False
)

fig.show()

Statistics of engineered features:
----------------------------------------
       rooms_per_household  bedrooms_per_room  population_per_household  \
count         19648.000000       19648.000000              19648.000000   
mean              5.361708           0.215716                  3.096560   
std               2.293321           0.064626                 10.639195   
min               0.846154           0.037151                  0.692308   
25%               4.416667           0.177476                  2.446614   
50%               5.185730           0.204543                  2.837779   
75%               5.971083           0.241258                  3.306021   
max             132.533333           2.824675               1243.333333   

       rooms_per_person  
count      19648.000000  
mean           1.938814  
std            1.100585  
min            0.002547  
25%            1.498461  
50%            1.911020  
75%            2.248384  
max           55.222222  


Using the interactive box plots above:
1. Which engineered features show the most variation?
2. Are there notable outliers in any of the features?
3. How do the median values compare between features?
4. Which features might be most useful for predicting house prices?

### 2.8. How should we handle categorical features?

We have one categorical feature (ocean_proximity) that needs to be encoded for our model.
What are the different ways we could encode this feature? What are the trade-offs?

YOUR ANSWER HERE

In [27]:
# Analyze categorical values
print("Distribution of ocean_proximity categories:")
print("-" * 40)
value_counts = df['ocean_proximity'].value_counts()
for category, count in value_counts.items():
    print(f"{category:20} {count:5d} ({count/len(df)*100:5.1f}%)")

# Create bar plot of category distribution
fig = px.bar(
    x=value_counts.index,
    y=value_counts.values,
    title='Distribution of Ocean Proximity Categories',
    labels={'x': 'Category', 'y': 'Count'},
    color=value_counts.values,
    color_continuous_scale='Viridis'
)

fig.update_layout(
    title_x=0.5,
    showlegend=False,
    template='plotly_white'
)

fig.show()

Distribution of ocean_proximity categories:
----------------------------------------
<1H OCEAN             8595 ( 43.7%)
INLAND                6523 ( 33.2%)
NEAR OCEAN            2437 ( 12.4%)
NEAR BAY              2088 ( 10.6%)
ISLAND                   5 (  0.0%)


Using the interactive bar chart above:
1. Which location category is most common?
2. Are there any rare categories we might want to handle differently?
3. How might the distribution of categories affect our model?

In [28]:
print("\nPerforming one-hot encoding...")
# YOUR CODE HERE
df = pd.get_dummies(df, columns=['ocean_proximity'], prefix='ocean')

print("\nNew features after one-hot encoding:")
ocean_cols = [col for col in df.columns if col.startswith('ocean_')]
for col in ocean_cols:
    print(f"- {col}")


Performing one-hot encoding...

New features after one-hot encoding:
- ocean_<1H OCEAN
- ocean_INLAND
- ocean_ISLAND
- ocean_NEAR BAY
- ocean_NEAR OCEAN


In [29]:
df.head(2)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,rooms_per_household,bedrooms_per_room,population_per_household,rooms_per_person,ocean_<1H OCEAN,ocean_INLAND,ocean_ISLAND,ocean_NEAR BAY,ocean_NEAR OCEAN
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,6.984127,0.146591,2.555556,2.732919,False,False,False,True,False
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,6.238137,0.155797,2.109842,2.956685,False,False,False,True,False


### 2.9 What is our final feature set?

In [30]:
print("Final Feature Set Analysis:")
print("-" * 40)
print(f"Total features: {len(df.columns)} (including target variable)")

# Group features by type
original_numeric = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 
                   'total_bedrooms', 'population', 'households', 'median_income']
engineered_numeric = ['rooms_per_household', 'bedrooms_per_room', 
                     'population_per_household', 'rooms_per_person']
categorical = [col for col in df.columns if col.startswith('ocean_')]
target = ['median_house_value']

print("\nFeature Groups:")
print("1. Original Numeric Features:")
for f in original_numeric:
    print(f"   - {f}")
print("\n2. Engineered Numeric Features:")
for f in engineered_numeric:
    print(f"   - {f}")
print("\n3. Encoded Categorical Features:")
for f in categorical:
    print(f"   - {f}")
print("\n4. Target Variable:")
print(f"   - {target[0]}")

# Create correlation matrix visualization
numeric_features = original_numeric + engineered_numeric
correlation_matrix = df[numeric_features + target].corr()

# Create interactive correlation heatmap
fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    text=np.round(correlation_matrix, 3),
    texttemplate='%{text}',
    textfont={"size": 10},
    hoverongaps=False,
    colorscale='RdBu',
    zmid=0
))

fig.update_layout(
    title='Feature Correlations',
    title_x=0.5,
    width=800,
    height=800,
    template='plotly_white'
)

fig.show()

# Print strongest correlations with target variable
target_correlations = correlation_matrix['median_house_value'].sort_values(ascending=False)
print("\nFeature correlations with median_house_value:")
for feature, corr in target_correlations.items():
    if feature != 'median_house_value':
        print(f"{feature:25} {corr:7.3f}")

Final Feature Set Analysis:
----------------------------------------
Total features: 18 (including target variable)

Feature Groups:
1. Original Numeric Features:
   - longitude
   - latitude
   - housing_median_age
   - total_rooms
   - total_bedrooms
   - population
   - households
   - median_income

2. Engineered Numeric Features:
   - rooms_per_household
   - bedrooms_per_room
   - population_per_household
   - rooms_per_person

3. Encoded Categorical Features:
   - ocean_<1H OCEAN
   - ocean_INLAND
   - ocean_ISLAND
   - ocean_NEAR BAY
   - ocean_NEAR OCEAN

4. Target Variable:
   - median_house_value



Feature correlations with median_house_value:
median_income               0.647
rooms_per_person            0.162
total_rooms                 0.145
rooms_per_household         0.112
households                  0.096
total_bedrooms              0.075
housing_median_age          0.065
population                  0.014
population_per_household   -0.021
longitude                  -0.046
latitude                   -0.149
bedrooms_per_room          -0.200


Using the interactive correlation heatmap above:
1. Which features have the strongest positive correlation with house value?
2. Are there any features that show strong correlations with each other?
3. Which features show little to no correlation with house value?
4. What might the correlation between total_rooms and total_bedrooms tell us?
5. Which engineered features show the strongest correlations with the target?

## 3. Model Preparation

Now that we understand our data and have engineered our features, let's prepare for modeling.

### 3.1. How should we split our data for training and evaluation?

Before we start modeling, we need to split our data into training, validation, and test sets.
Think about:
1. What split ratios should we use?
2. Why do we need a validation set in addition to a test set?
3. Should we stratify our splits for regression?

YOUR ANSWER HERE

In [36]:
# Import scikit-learn modules for modeling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# Prepare features and target
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

# YOUR CODE HERE
# Random split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Stratify
strata = pd.qcut(y, q=5, labels=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=strata)
strata = pd.qcut(y_train, q=5, labels=False)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=strata)

print("Data split sizes:")
print(f"Training set:   {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation set: {len(X_val)} samples ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test set:       {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")

Data split sizes:
Training set:   12574 samples (64.0%)
Validation set: 3144 samples (16.0%)
Test set:       3930 samples (20.0%)


### 3.2. How should we preprocess our features?

Different features have different scales and distributions. How should we handle this?

YOUR ANSWER HERE

In [37]:
# Import preprocessing tools
from sklearn.preprocessing import StandardScaler

# Initialize and fit scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert to DataFrames with column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

display(X_train.head(1))
display(X_train_scaled.head(1))

# Show statistics of scaled features
print("\nScaled feature statistics (training set):")
print(X_train_scaled.describe().round(3))

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,rooms_per_household,bedrooms_per_room,population_per_household,rooms_per_person,ocean_<1H OCEAN,ocean_INLAND,ocean_ISLAND,ocean_NEAR BAY,ocean_NEAR OCEAN
15840,-122.43,37.75,52.0,2960.0,623.0,1191.0,589.0,3.95,5.025467,0.210473,2.022071,2.485306,False,False,False,True,False


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,rooms_per_household,bedrooms_per_room,population_per_household,rooms_per_person,ocean_<1H OCEAN,ocean_INLAND,ocean_ISLAND,ocean_NEAR BAY,ocean_NEAR OCEAN
0,-1.434358,0.978293,1.893266,0.170768,0.218935,-0.213286,0.243979,0.172797,-0.139562,-0.08231,-0.420392,0.457213,-0.891917,-0.700748,-0.012613,2.908823,-0.372344



Scaled feature statistics (training set):
       longitude   latitude  housing_median_age  total_rooms  total_bedrooms  \
count  12574.000  12574.000           12574.000    12574.000       12574.000   
mean       0.000      0.000              -0.000       -0.000           0.000   
std        1.000      1.000               1.000        1.000           1.000   
min       -2.395     -1.444              -2.201       -1.233          -1.310   
25%       -1.099     -0.801              -0.836       -0.550          -0.576   
50%        0.531     -0.643              -0.033       -0.230          -0.244   
75%        0.782      0.969               0.689        0.243           0.268   
max        2.547      2.935               1.893       14.253          14.549   

       population  households  median_income  rooms_per_household  \
count   12574.000   12574.000      12574.000            12574.000   
mean       -0.000       0.000         -0.000                0.000   
std         1.000       1.000

## 4. Model Training and Evaluation

### 4.1. How do we evaluate regression models?

Before we start training models, let's understand how to evaluate their performance.
What metrics should we use for regression problems?

YOUR ANSWER HERE

### 4.2. What models should we try?

Let's start with the simplest regression model: linear regression. We'll explore other models in future weeks.

In [38]:
# Import models
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Initialize and train linear regression
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)

# Make predictions
y_train_pred = linear_model.predict(X_train_scaled)
y_val_pred = linear_model.predict(X_val_scaled)

# Calculate metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)

print("Linear Regression Results:")
print("-" * 30)
print(f"Training RMSE: ${train_rmse:,.2f}")
print(f"Validation RMSE: ${val_rmse:,.2f}")
print(f"Training R²: {train_r2:.4f}")
print(f"Validation R²: {val_r2:.4f}")

# Create scatter plot of predictions
fig = make_subplots(rows=1, cols=2, subplot_titles=['Training Predictions', 'Validation Predictions'])

# Training predictions
fig.add_trace(
    go.Scatter(x=y_train, y=y_train_pred, mode='markers', name='Training',
               marker=dict(color='lightcoral', opacity=0.6)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=[y_train.min(), y_train.max()], 
               y=[y_train.min(), y_train.max()],
               mode='lines', name='Perfect Predictions',
               line=dict(color='black', dash='dash')),
    row=1, col=1
)

# Validation predictions
fig.add_trace(
    go.Scatter(x=y_val, y=y_val_pred, mode='markers', name='Validation',
               marker=dict(color='lightblue', opacity=0.6)),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=[y_val.min(), y_val.max()], 
               y=[y_val.min(), y_val.max()],
               mode='lines', showlegend=False,
               line=dict(color='black', dash='dash')),
    row=1, col=2
)

# Update layout
fig.update_layout(
    height=500,
    title_text='Linear Regression: Actual vs Predicted Values',
    title_x=0.5,
    template='plotly_white'
)

# Update axes
for i in [1, 2]:
    fig.update_xaxes(title_text='Actual Values ($)', row=1, col=i)
    fig.update_yaxes(title_text='Predicted Values ($)', row=1, col=i)

fig.show()

Linear Regression Results:
------------------------------
Training RMSE: $58,362.04
Validation RMSE: $59,272.41
Training R²: 0.6377
Validation R²: 0.6322


Using the scatter plots above:
1. How well does the model fit the training data?
2. Is there evidence of underfitting or overfitting?
3. Where does the model make its largest errors?
4. How do the training and validation results compare?
5. What might explain the model's performance?